# Preamble

In [ ]:
# Copy the wheel file from your bucket to the cluster's local storage
!gsutil cp gs://mksyunz-sparkly-bucket/spark-sql/sparksql_magic*.whl /tmp/

# Install the package directly from the local file
%pip install /tmp/sparksql_magic*.whl

In [ ]:
%load_ext sparksql_magic

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, LongType, StringType, DoubleType

# On Dataproc, this connects to the existing cluster manager
spark = SparkSession.builder.getOrCreate()
SPANNER_OPTS = {
    "projectId":  "improvingvancouver",
    "instanceId": "mksyunz-spark-dev",
    "databaseId": "demo",
}

# This grabs the underlying JVM logger to silence it completely
log4jLogger = spark._jvm.org.apache.log4j
log = log4jLogger.LogManager.getLogger("org")
log.setLevel(log4jLogger.Level.ERROR)

spark.sparkContext.setLogLevel("ERROR")

## Register the Spanner catalog 

In [ ]:
spark.conf.set("spark.sql.catalog.spanner",
               "com.google.cloud.spark.spanner.SpannerCatalog")
spark.conf.set("spark.sql.catalog.spanner.projectId",  SPANNER_OPTS["projectId"])
spark.conf.set("spark.sql.catalog.spanner.instanceId", SPANNER_OPTS["instanceId"])
spark.conf.set("spark.sql.catalog.spanner.databaseId", SPANNER_OPTS["databaseId"])


print(f"Spanner: {SPANNER_OPTS['projectId']}/{SPANNER_OPTS['instanceId']}/{SPANNER_OPTS['databaseId']}")
print("✓ Spanner catalog registered.")

In [ ]:
%%sparksql
DROP TABLE IF EXISTS spanner.demo_json

# Spark Spanner Connector — JSON Demo 

This notebook demonstrates how to read and write JSON content using Spark Spanner connector

Now, if row with same `id` is inserted, the save will fail:

Spanner `JSON` columns map to Spark `StringType`. 

To write structured data as JSON to Spanner:
1. Build a DataFrame with a struct column
2. Convert the struct to a JSON string with `to_json()`
3. Write the string to the Spanner JSON column

To read it back, use `from_json()` to parse the JSON string into a struct.

## A. Create a table with a JSON column

In [ ]:
%%sparksql
CREATE TABLE spanner.demo_json (
    id       BIGINT NOT NULL,
    name     STRING,
    metadata STRING
) USING `cloud-spanner`
TBLPROPERTIES('primaryKeys' = 'id')

## B. Build a DataFrame with a struct column and convert to JSON

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, ArrayType

# Define a schema with a nested struct column
nested_schema = StructType([
    StructField("id",   LongType(), False),
    StructField("name", StringType(), True),
    StructField("metadata", StructType([
        StructField("role",       StringType(), True),
        StructField("level",      IntegerType(), True),
        StructField("skills",     ArrayType(StringType()), True),
    ]), True),
])

data = [
    (1, "Alice",   ("engineer",  5, ["python", "spark", "sql"])),
    (2, "Bob",     ("analyst",   3, ["sql", "tableau"])),
    (3, "Charlie", ("manager",   7, ["leadership", "strategy"])),
]
struct_df = spark.createDataFrame(data, nested_schema)

print("Original DataFrame with struct column:")
struct_df.printSchema()
struct_df.show(truncate=False)

In [ ]:
# Convert the struct column to a JSON string for writing to Spanner
write_df = struct_df.withColumn("metadata", F.to_json("metadata"))

print("DataFrame after to_json() — ready to write to Spanner:")
write_df.printSchema()
write_df.show(truncate=False)

## C. Write to Spanner

In [ ]:
(write_df.write.format("cloud-spanner")
    .options(**SPANNER_OPTS)
    .option("table", "demo_json")
    .mode("append")
    .save())

print("✓ Wrote 3 rows with JSON metadata to Spanner.")

## D. Read back — raw JSON strings

In [ ]:
%%sparksql
SELECT * FROM spanner.demo_json ORDER BY id

## E. Parse JSON back into structs with `from_json()`

### Read via DataFrame API

In [ ]:
raw_df = (spark.read.format("cloud-spanner")
    .options(**SPANNER_OPTS)
    .option("table", "demo_json")
    .load()
    .orderBy("id"))

In [ ]:
### Define the expected struct schema for parsing

In [ ]:
json_schema = StructType([
    StructField("role",       StringType(), True),
    StructField("level",      IntegerType(), True),
    StructField("skills",     ArrayType(StringType()), True),
])

In [ ]:
### Parse the JSON string back into a struct

In [ ]:
parsed_df = raw_df.withColumn("metadata", F.from_json("metadata", json_schema))

print("Parsed DataFrame — struct column restored:")
parsed_df.printSchema()
parsed_df.show(truncate=False)

In [ ]:
### Access nested fields directly

In [ ]:
parsed_df.select(
    "id",
    "name",
    F.col("metadata.role").alias("role"),
    F.col("metadata.level").alias("level"),
    F.explode("metadata.skills").alias("skill"),
).show(truncate=False)

---
## Cleanup

In [ ]:
%%sparksql
DROP TABLE IF EXISTS spanner.demo_json